> **You do not need to run this notebook to reproduce the results.**
> It builds the annotation samples from the cleaned corpus (`data/processed/comments_clean.csv`), which is **not distributed** (excluded for size). The completed manual annotations are provided as `data/processed/labelled_comments_final.csv`, and the reproducible pipeline runs from **notebook 04** onward. See README §6–§7.

# 03 — Sampling for annotation (RQ1)

Selects comments for manual annotation from the cleaned corpus, in two stages:

1. **Channel-stratified sample** — per channel, 60% from a keyword-matched
   stratum and 40% from the remainder, interleaved across channels.
2. **Supplementary keyword screen** — flags likely weaponised-victimhood (1B)
   and conspiratorial-framing (2) comments not already selected, to enrich the
   rarer constructs for training.

Seed 42 throughout. The final annotated set is provided in
`data/processed/labelled_comments_final.csv`.

In [ ]:
import re
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEED, N_PER_CHANNEL, KEYWORD_FRAC = 42, 500, 0.6

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "comments_clean.csv")

# Flag comments containing manosphere-relevant vocabulary
KEYWORD_PATTERN = r"""
\b(?: men|male|males|boys|masculinity|women|female|females|girls|
    feminism|feminist|misandry|misandrist|woke|leftist|liberal|progressive|sjw|
    red\s*pill|mgtow|gynocentric|hypergamy|simp|beta|alpha|
    marriage|divorce|family\s*court|custody|child\s*support|alimony|
    victim|victims|oppressed|oppression|discriminated|discrimination|
    unfair|rigged|stacked|double\s*standard|demonized|demonised|
    blamed|attacked|hated|punished|destroyed|
    agenda|propaganda|brainwashed|indoctrinated|matrix|
    narrative|elite|globalist|government|system|media|schools )\b
"""
df["keyword_relevant"] = df["text_clean"].fillna("").str.contains(
    KEYWORD_PATTERN, flags=re.IGNORECASE | re.VERBOSE, regex=True
)

# Channel-stratified sample: 60% keyword-matched / 40% remainder
parts = []
for channel, group in df.groupby("channel_handle"):
    n_kw = int(round(N_PER_CHANNEL * KEYWORD_FRAC))
    kw_pool, rand_pool = group[group.keyword_relevant], group[~group.keyword_relevant]

    take_kw   = kw_pool.sample(n=min(n_kw, len(kw_pool)), random_state=SEED).assign(sample_source="keyword")
    take_rand = rand_pool.sample(n=min(N_PER_CHANNEL - n_kw, len(rand_pool)), random_state=SEED).assign(sample_source="random")
    chosen = pd.concat([take_kw, take_rand]).drop_duplicates("comment_id")

    target = min(N_PER_CHANNEL, len(group))
    if len(chosen) < target:
        remainder = group[~group.comment_id.isin(chosen.comment_id)]
        top = remainder.sample(n=target - len(chosen), random_state=SEED).assign(sample_source="top_up")
        chosen = pd.concat([chosen, top])
    chosen["channel_order"] = range(len(chosen))
    parts.append(chosen)

# Interleave channels so any stopping point stays balanced
stage1 = pd.concat(parts).sort_values(["channel_order", "channel_handle"]).reset_index(drop=True)

keep = ["comment_id", "channel_handle", "text_clean", "is_reply", "keyword_relevant", "sample_source"]
label = stage1[keep].copy()
for col in ["label_1A", "label_1B", "label_2", "notes"]:
    label[col] = ""
label.to_csv(PROJECT_ROOT / "data" / "processed" / "annotation_labelling.csv", index=False)
print(f"Stage 1: {len(label)} comments -> annotation_labelling.csv")

In [ ]:
# Stage 2 — supplementary keyword screen for the two rarer constructs

KW_1B = [
    # feminism / women blamed for harm
    "feminism destroyed", "feminism ruined", "feminism broke", "feminism created",
    "thanks to feminism", "because of feminism", "feminism is to blame",
    "feminism is the problem", "feminism is cancer", "feminazi", "nazi feminism",
    "feminists did this", "feminists did", "feminists caused", "feminists are to blame",
    "feminists are the problem", "women destroyed", "women ruined", "women broke",
    "women did this", "women caused this", "this is what women wanted",
    "this is what you wanted", "you wanted equality", "you voted for this",
    "their fault", "women's fault", "its your fault women", "your fault ladies",
    "look what they've done", "ruined dating", "ruined marriage", "ruined relationships",
    # men as victims / aggrieved entitlement
    "men are the real victims", "men are the victims", "men are victims too",
    "men get nothing", "men gets nothing", "men get blamed", "men are blamed",
    "war on men", "war against men", "society hates men", "society is against men",
    "system against men", "anti male", "anti-male", "anti man", "anti-man",
    "misandry", "misandrist", "misandric", "men are disposable", "men are used",
    "used and discarded", "no one cares about men", "nobody cares about men",
    # deflection / outgroup plays victim
    "playing the victim", "play the victim", "playing victim", "victim card",
    "crocodile tears", "always the victim", "never take accountability",
    "no accountability", "zero accountability", "never their fault",
    "never held accountable", "can do no wrong", "women can't be held",
    # mobilisation / withdrawal / MGTOW
    "mgtow", "m.g.t.o.w", "go their own way", "going your own way",
    "men go their own way", "men going their own way", "go your own way",
    "don't get married", "never get married", "never marry", "do not marry",
    "avoid marriage", "avoid getting married", "stay single", "remain single",
    "opt out", "check out of society", "drop out of society", "walk away",
    "walk away from women", "walk away from marriage", "stop dating", "stop chasing",
    "stop simping", "stop being a simp", "no wife no kids", "no marriage no kids",
    # marriage as trap / no incentive
    "marriage is a scam", "marriage is slavery", "marriage is a trap",
    "marriage is dead", "marriage is obsolete", "no incentive", "zero incentive",
    "nothing to gain", "juice isn't worth the squeeze", "why would any man",
    "what's the point of marriage",
    # economic withdrawal / strike framing
    "stop paying", "stop providing", "stop protecting", "keep your money",
    "protect your money", "protect your assets", "men on strike", "male strike",
    "go on strike", "take back control", "take back power",
    # punitive / rights-removal
    "hold them accountable", "hold women accountable", "make women accountable",
    "make them pay", "strip women of", "strip women from", "strip them of",
    "remove their rights", "take their rights away", "stop them from voting",
    "revoke women's rights", "reverse women's rights", "repeal the 19th",
    "19th amendment", "women shouldn't vote", "return to patriarchy",
    "bring back patriarchy",
    # radicalisation framing
    "this will radicalise", "this will radicalize", "creating incels",
    "pushed too far", "nothing left to lose", "reckoning is coming", "backlash",
    # collective mobilisation frames
    "men need to", "men must", "men have to", "we men need to", "we men must",
    "we need to", "we must", "we have to", "we as men", "as men we", "stand up",
    "fight back", "push back", "wake up men", "men wake up", "wake up brothers",
    "stick together", "brothers need to", "spread the word",
    # male supremacy / contribution tied to grievance
    "real men", "be a real man", "men built this", "men built everything",
    "we built this", "without men", "if men disappeared", "men deserve better",
    "men deserve respect", "men deserve more",
    # red-pill jargon carrying implicit 1B
    "awalt", "all women are like that", "briffault", "hypergamy", "hypergamous",
    "the wall", "hit the wall", "hitting the wall", "post wall", "alpha widow",
    "cash and prizes", "divorce rape", "married to the state",
    "government is the husband", "female nature", "women's nature",
    "that's what women do", "female privilege", "women have it easy", "easy mode",
    "double standard", "double standards", "80/20", "the red pill", "red pilled",
    "took the red pill", "unplug",
]

KW_2 = [
    # deliberate design / intent
    "by design", "not an accident", "no accident", "no coincidence", "on purpose",
    "deliberate", "deliberately", "intentional", "intentionally", "orchestrated",
    "engineered", "manufactured", "planned", "it was planned", "the plan",
    "part of the plan", "all part of", "the agenda", "hidden agenda", "the real agenda",
    "their agenda", "pushing an agenda", "social engineering", "they want you to",
    "they want us", "the goal is", "their goal", "the real goal", "designed to",
    "meant to", "set up to", "created to", "intended to",
    # covert / hidden / suppressed truth
    "behind the scenes", "in the shadows", "cover up", "covering up", "covered up",
    "hide the truth", "hiding the truth", "they don't want you to know",
    "don't want you to know", "suppress", "suppressed", "suppressing", "censored",
    "censoring", "shadow ban", "the truth about", "official narrative", "lied to us",
    "been lied to", "wake up sheep", "sheeple", "open your eyes",
    "controlled opposition", "gatekeepers", "false flag",
    # powerful / malevolent actors
    "elites", "the elite", "globalist", "globalists", "the establishment",
    "powers that be", "deep state", "the matrix", "new world order",
    "world economic forum", "united nations", "rothschild", "soros", "banksters",
    "central bank", "big pharma", "mainstream media", "the media", "hollywood",
    "the government wants", "feminists lobbied", "the sisterhood", "cultural marxism",
    "cultural marxist", "communist agenda", "satanic", "the enemy", "they control",
    # manipulation of the masses
    "brainwash", "brainwashed", "brainwashing", "indoctrinate", "indoctrinated",
    "indoctrination", "propaganda", "programmed", "programming", "conditioned",
    "psyop", "manipulate", "manipulated", "manipulating", "mind control", "puppet",
    "puppets", "useful idiots", "media pushes", "media is pushing",
    # malevolent action
    "trying to destroy", "out to destroy", "want to destroy", "designed to destroy",
    "destroy the family", "destroy masculinity", "destroy society", "destroy men",
    "tear down", "break down the family", "dismantle", "undermine", "erode",
    "erase men", "replace men", "great replacement", "depopulation",
    # generic 'rigged system'
    "the system is rigged", "system is rigged", "rigged system", "the whole system",
    "the system is", "the game is rigged", "rigged game", "it's all rigged",
    "corrupt system", "the system is corrupt",
]

def make_pattern(terms):
    return re.compile(r"(?<!\w)(?:" + "|".join(re.escape(t) for t in terms) + r")(?!\w)", re.I)

txt = df["text_clean"].fillna("")
df["hit_1B"] = txt.str.contains(make_pattern(KW_1B))
df["hit_2"]  = txt.str.contains(make_pattern(KW_2))
print("1B hits:", int(df.hit_1B.sum()), "| 2 hits:", int(df.hit_2.sum()),
      "| either:", int((df.hit_1B | df.hit_2).sum()))

# Exclude comments already chosen in Stage 1
already = set(label["comment_id"])
candidates = df[(df.hit_1B | df.hit_2) & ~df.comment_id.isin(already)].copy()

# ~300 from each construct flag, de-duplicated into one supplementary batch
b1 = candidates[candidates.hit_1B].sample(min(300, int(candidates.hit_1B.sum())), random_state=SEED)
b2 = candidates[candidates.hit_2].sample(min(300, int(candidates.hit_2.sum())), random_state=SEED)
batch = pd.concat([b1, b2]).drop_duplicates("comment_id").reset_index(drop=True)
for col in ["label_1A", "label_1B", "label_2", "notes"]:
    batch[col] = ""

out = batch[["comment_id", "channel_handle", "text_clean", "hit_1B", "hit_2",
             "label_1A", "label_1B", "label_2", "notes"]]
out.to_csv(PROJECT_ROOT / "data" / "processed" / "keyword_candidates.csv", index=False)
print(f"Stage 2: {len(out)} supplementary candidates -> keyword_candidates.csv")

In [ ]:
print(label["sample_source"].value_counts())
print(label["sample_source"].value_counts(normalize=True).mul(100).round(1))